# Notebook 01: Data Collection

**Project:** City of Boston – Building & Housing Violations Analysis  
**Group:** Mengxing Wang, Jiacheng He, Xiao Luo, Renwei Li  

## Overview

This notebook handles data acquisition. The primary dataset is the Boston Building and Property Violations dataset, which is publicly available via Boston's Open Data portal. We also document the neighborhood population data sourced from the U.S. Census Bureau, which will be used in Notebook 04 for normalization.

**Data sources:**
- Boston Building & Property Violations: https://data.boston.gov/
- Boston Population Estimates: https://data.boston.gov/dataset/2025-boston-population-estimates-neighborhood-level

In [1]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path

# Project root relative to this notebook
PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'

for d in [RAW_DIR, PROCESSED_DIR, EXTERNAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directory structure verified.')
print(f'Raw data dir: {RAW_DIR}')

Directory structure verified.
Raw data dir: D:\Projects\CS506Project\data\raw


## 1. Load the Violations Dataset

The raw CSV was downloaded from the Boston Open Data portal and placed in `data/raw/violations.csv`. We load and inspect it here.

In [2]:
violations_path = RAW_DIR / 'violations.csv'
df = pd.read_csv(violations_path, low_memory=False)

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Shape: (17172, 24)
Columns: ['case_no', 'ap_case_defn_key', 'status_dttm', 'status', 'code', 'value', 'description', 'violation_stno', 'violation_sthigh', 'violation_street', 'violation_suffix', 'violation_city', 'violation_state', 'violation_zip', 'ward', 'contact_addr1', 'contact_addr2', 'contact_city', 'contact_state', 'contact_zip', 'sam_id', 'latitude', 'longitude', 'location']


,case_no,ap_case_defn_key,status_dttm,status,code,value,description,violation_stno,violation_sthigh,violation_street,...,ward,contact_addr1,contact_addr2,contact_city,contact_state,contact_zip,sam_id,latitude,longitude,location
0,V91983,1013,NaN,Closed,121.2,NaN,Unsafe and Dangerous,302,NaN,Sumner,...,01,302 Sumner St,NaN,East Boston,MA,02128,132380.0,42.367679,-71.03658,"(42.36767899977675, -71.03658000178699)"
1,V897990,1013,2026-03-20 15:27:46,Open,107.4,NaN,Failed to comply w permit term,14,NaN,Elder,...,07,58 GAINSBOROUGH ST,NaN,BOSTON,MA,02115,52584.0,42.320230,-71.06334,"(42.32023000030727, -71.06334000096946)"
2,V897956,1013,2026-03-20 10:37:20,Open,102.8,NaN,Maintenance,111,NaN,Quincy,...,13,111 QUINCY ST,NaN,DORCHESTER,MA,02121,113997.0,42.313830,-71.07762,"(42.3138299994726, -71.07762000133053)"


## 2. Initial Data Inspection

We check column data types, missing value rates, and the date range covered by the dataset.

In [3]:
print('--- Data Types ---')
print(df.dtypes)
print()
print('--- Missing Values (%) ---')
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(missing_pct[missing_pct > 0])

--- Data Types ---
case_no                 str
ap_case_defn_key      int64
status_dttm             str
status                  str
code                    str
value               float64
description             str
violation_stno          str
violation_sthigh        str
violation_street        str
violation_suffix        str
violation_city          str
violation_state         str
violation_zip           str
ward                    str
contact_addr1           str
contact_addr2           str
contact_city            str
contact_state           str
contact_zip             str
sam_id              float64
latitude            float64
longitude           float64
location                str
dtype: object

--- Missing Values (%) ---
status_dttm           0.01
value               100.00
description           1.44
violation_sthigh     75.54
violation_suffix      0.84
contact_addr1         0.03
contact_addr2        82.16
contact_city          0.01
contact_state         0.01
contact_zip           0.

In [4]:
# Parse the status date column
df['status_dttm'] = pd.to_datetime(df['status_dttm'], errors='coerce')
valid_dates = df['status_dttm'].dropna()
print(f'Date range: {valid_dates.min()} → {valid_dates.max()}')
print(f'Records with valid date: {valid_dates.shape[0]:,} of {len(df):,}')

Date range: 2009-12-01 13:18:47 → 2026-03-20 15:27:46
Records with valid date: 17,171 of 17,172


## 3. Neighborhood Population Data (External)

We use official neighborhood population estimates from the City of Boston 
Planning Department Research Division. The data is downloaded from the 
Boston Open Data Portal (data.boston.gov)

In [11]:
pop_df = pop_raw[['name', 'population_b03002_001e']].copy()
pop_df = pop_df.rename(columns={
    'name': 'neighborhood',
    'population_b03002_001e': 'population_2025'  # ← 改这里
})

pop_df = pop_df.dropna(subset=['population_2025'])
pop_df['population_2025'] = pop_df['population_2025'].astype(int)
pop_df.to_csv(EXTERNAL_DIR / 'neighborhood_population.csv', index=False)
print('Saved!')
print(pop_df)

Saved!
               neighborhood  population_2025
0                   Allston            31810
1                  Back Bay            18983
2               Beacon Hill             9327
3                  Brighton            55869
4               Charlestown            19232
5                 Chinatown             6371
6                Dorchester           123056
7                  Downtown            15752
8               East Boston            46892
9                    Fenway            42351
10                Hyde Park            33469
11            Jamaica Plain            42949
12                 Longwood             5553
13                 Mattapan            24424
14             Mission Hill            19000
15                North End            10635
16               Roslindale            29378
17                  Roxbury            56800
18             South Boston            40904
19  South Boston Waterfront             8289
20                South End            34582
21 

## 4. Unique Values in Key Columns

We examine the range of violation codes, statuses, and cities to understand what the dataset covers.

In [9]:
print('Unique statuses:', df['status'].unique())
print(f'Unique violation codes: {df["code"].nunique()}')
print(f'Unique violation descriptions: {df["description"].nunique()}')
print('Top 10 cities in data:')
print(df['violation_city'].value_counts().head(10))

Unique statuses: <StringArray>
['Closed', 'Open', 'Void']
Length: 3, dtype: str
Unique violation codes: 520
Unique violation descriptions: 419
Top 10 cities in data:
violation_city
Dorchester      4687
Boston          2787
East Boston     1770
Roxbury         1656
Mattapan         889
South Boston     875
Hyde Park        834
Brighton         762
Allston          611
Roslindale       556
Name: count, dtype: int64


## 5. Summary

Key findings from data collection:
- The dataset contains **~17,000 violation records** spanning multiple years.
- Key fields include violation code, description, city/neighborhood, status, and geo-coordinates.
- A meaningful fraction of records lack a `status_dttm` (date), which we address in Notebook 02.
- External population data has been saved to `data/external/neighborhood_population.csv` for later normalization.

In [10]:
# Save a summary metadata file
summary = {
    'total_records': int(len(df)),
    'columns': df.columns.tolist(),
    'date_min': str(valid_dates.min()),
    'date_max': str(valid_dates.max()),
    'unique_codes': int(df['code'].nunique()),
}
with open(PROCESSED_DIR / 'dataset_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Summary saved to data/processed/dataset_summary.json')

Summary saved to data/processed/dataset_summary.json
